<a href="https://colab.research.google.com/github/personallypetra/Grand_Challenge-/blob/Wind-%26-Factories/Wind.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Wind Data

In [ ]:
!pip install openmeteo-requests
!pip install requests-cache retry-requests numpy pandas
!pip install openmeteo-requests requests-cache retry-requests

In [ ]:
"""
Pull hourly wind data from Open-Meteo API
==========================================
10 Lazio stations, 2018-01-01 to 2024-12-31

pip install openmeteo-requests requests-cache retry-requests
"""

import openmeteo_requests
import pandas as pd
import requests_cache
from retry_requests import retry
import time

cache_session = requests_cache.CachedSession('.cache', expire_after=3600)
retry_session = retry(cache_session, retries=5, backoff_factor=0.2)
openmeteo = openmeteo_requests.Client(session=retry_session)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
OUTPUT_PATH = "/content/wind_data_lazio.csv"

stations = {
    "ARENULA":          {"lat": 41.894,  "lon": 12.4754},
    "BUFALOTTA":        {"lat": 41.9477, "lon": 12.5337},
    "C.SO FRANCIA":     {"lat": 41.9474, "lon": 12.4696},
    "CINECITTA":        {"lat": 41.8577, "lon": 12.5687},
    "CIPRO":            {"lat": 41.9064, "lon": 12.4476},
    "FERMI":            {"lat": 41.864,  "lon": 12.4696},
    "L.GO MAGNA GRECIA":{"lat": 41.8831, "lon": 12.509},
    "L.GO PERESTRELLO": {"lat": 41.886,  "lon": 12.5416},
    "TIBURTINA":        {"lat": 41.9103, "lon": 12.5489},
    "VILLA ADA":        {"lat": 41.9329, "lon": 12.5069},
}

START_DATE = "2018-01-01"
END_DATE = "2024-12-31"

all_dfs = []

for name, coords in stations.items():
    print(f"Fetching {name}...")

    try:
        params = {
            "latitude": coords["lat"],
            "longitude": coords["lon"],
            "start_date": START_DATE,
            "end_date": END_DATE,
            "hourly": ["wind_speed_10m", "wind_direction_10m"],
            "timezone": "Europe/Rome",
        }

        responses = openmeteo.weather_api(ARCHIVE_URL, params=params)
        response = responses[0]

        hourly = response.Hourly()
        wind_speed = hourly.Variables(0).ValuesAsNumpy()
        wind_direction = hourly.Variables(1).ValuesAsNumpy()

        dates = pd.date_range(
            start=pd.to_datetime(hourly.Time(), unit="s", utc=True),
            end=pd.to_datetime(hourly.TimeEnd(), unit="s", utc=True),
            freq=pd.Timedelta(seconds=hourly.Interval()),
            inclusive="left",
        )

        df = pd.DataFrame({
            "date": dates,
            "wind_speed_10m": wind_speed,
            "wind_direction_10m": wind_direction,
            "station": name,
            "latitude": coords["lat"],
            "longitude": coords["lon"],
        })

        all_dfs.append(df)
        print(f"  {len(df):,} rows")

    except Exception as e:
        print(f"  ERROR: {e}")

    time.sleep(1)

print("\nCombining...")
df_wind = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows: {df_wind.shape[0]:,}")
print(f"Date range: {df_wind['date'].min()} to {df_wind['date'].max()}")

df_wind.to_csv(OUTPUT_PATH, index=False)
print(f"Saved to {OUTPUT_PATH}")

In [ ]:
wind_df = pd.read_csv("wind_data_lazio.csv")

In [ ]:
# ============================================================
print("=" * 60)
print("1. BASIC STRUCTURE")
print("=" * 60)
print(f"Shape: {wind_df.shape[0]:,} rows x {wind_df.shape[1]} columns")
print(f"\nColumns: {wind_df.columns.tolist()}")
print(f"\nData types:")
print(wind_df.dtypes)
print(f"\nFirst 5 rows:")
print(wind_df.head())


In [ ]:
# ============================================================
# 2. SUMMARY STATISTICS
# ============================================================
print("\n" + "=" * 60)
print("2. SUMMARY STATISTICS")
print("=" * 60)
print(wind_df.describe())

In [ ]:
print("\n" + "=" * 60)
print("3. MISSING DATA")
print("=" * 60)
nulls = wind_df.isna().sum()
null_pct = (wind_df.isna().sum() / len(wind_df) * 100).round(2)
missing = pd.DataFrame({"count": nulls, "percent": null_pct}).sort_values("percent", ascending=False)
print(missing.to_string())

In [ ]:
# 5. DATE RANGE
# ============================================================
print("\n" + "=" * 60)
print("5. DATE RANGE")
print("=" * 60)
wind_df["date"] = pd.to_datetime(wind_df["date"])
wind_df["Year"] = wind_df["date"].dt.year
print(f"Date range: {wind_df['date'].min()} to {wind_df['date'].max()}")
print(f"\nRecords per year:")
print(wind_df["Year"].value_counts().sort_index())

In [ ]:
# 6a. Wind speed distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

wind_df["wind_speed_10m"].dropna().plot(kind="hist", bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Wind Speed Distribution")
axes[0].set_xlabel("Wind Speed (m/s)")

wind_df["wind_direction_10m"].dropna().plot(kind="hist", bins=36, ax=axes[1], color="darkorange", edgecolor="white")
axes[1].set_title("Wind Direction Distribution")
axes[1].set_xlabel("Direction (degrees)")

plt.tight_layout()
plt.show()


In [ ]:
import contextily as ctx
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from io import BytesIO

stations_info = wind_df.groupby("station").agg({"latitude": "first", "longitude": "first"}).reset_index()
direction_bins = np.linspace(0, 360, 37)

fig, ax_map = plt.subplots(figsize=(14, 14))

# Set map bounds with padding
pad = 0.015
ax_map.set_xlim(stations_info["longitude"].min() - pad, stations_info["longitude"].max() + pad)
ax_map.set_ylim(stations_info["latitude"].min() - pad, stations_info["latitude"].max() + pad)

# Add basemap
ctx.add_basemap(ax_map, crs="EPSG:4326", source=ctx.providers.OpenStreetMap.Mapnik)

# For each station, create a mini wind rose and paste it on the map
for _, row in stations_info.iterrows():
    station = row["station"]
    lat, lon = row["latitude"], row["longitude"]

    # Create mini wind rose as an image
    fig_mini, ax_mini = plt.subplots(figsize=(4, 4), subplot_kw={"projection": "polar"})
    data = wind_df[wind_df["station"] == station]["wind_direction_10m"].dropna()
    counts, _ = np.histogram(data, bins=direction_bins)
    theta = np.deg2rad((direction_bins[:-1] + direction_bins[1:]) / 2)
    ax_mini.bar(theta, counts, width=np.deg2rad(10), color="red", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax_mini.set_theta_zero_location("N")
    ax_mini.set_theta_direction(-1)
    ax_mini.set_yticklabels([])
    ax_mini.set_xticklabels([])
    ax_mini.patch.set_alpha(0)
    fig_mini.patch.set_alpha(0)

    # Save mini figure to buffer
    buf = BytesIO()
    fig_mini.savefig(buf, format="png", dpi=80, bbox_inches="tight", transparent=True)
    plt.close(fig_mini)
    buf.seek(0)

    # Read image and place on map
    from PIL import Image
    img = Image.open(buf)
    img_array = np.array(img)

    im = OffsetImage(img_array, zoom=0.6)
    ab = AnnotationBbox(im, (lon, lat), frameon=False)
    ax_map.add_artist(ab)

    # Add station name
    ax_map.annotate(station, (lon, lat), fontsize=7, fontweight="bold",
                    ha="center", va="top", xytext=(0, -30), textcoords="offset points",
                    bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.8))

ax_map.set_title("Wind Direction by Station - Rome", fontsize=16, fontweight="bold")
ax_map.set_xlabel("Longitude")
ax_map.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# 6d. Monthly average wind speed
wind_df["month"] = wind_df["date"].dt.month
monthly = wind_df.groupby("month")["wind_speed_10m"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
monthly.plot(kind="bar", ax=ax, color="teal")
ax.set_title("Average Wind Speed by Month (All Stations)")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Wind Speed (m/s)")
plt.tight_layout()
plt.show()


In [ ]:

# 6e. Hourly pattern
wind_df["hour"] = wind_df["date"].dt.hour
hourly = wind_df.groupby("hour")["wind_speed_10m"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
hourly.plot(kind="line", ax=ax, marker="o", color="coral")
ax.set_title("Average Wind Speed by Hour of Day")
ax.set_xlabel("Hour")
ax.set_ylabel("Mean Wind Speed (m/s)")
plt.tight_layout()
plt.show()


In [ ]:
# 6f. Yearly average wind speed
yearly = wind_df.groupby("Year")["wind_speed_10m"].mean()

fig, ax = plt.subplots(figsize=(10, 5))
yearly.plot(kind="bar", ax=ax, color="mediumpurple")
ax.set_title("Average Wind Speed by Year")
ax.set_xlabel("Year")
ax.set_ylabel("Mean Wind Speed (m/s)")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
stations_sorted = sorted(wind_df["station"].unique())
colors = plt.cm.tab10(np.linspace(0, 1, len(stations_sorted)))
linestyles = ["-", "--", "-.", ":", "-", "--", "-.", ":", "-", "--"]

for i, station in enumerate(stations_sorted):
    sub = wind_df[wind_df["station"] == station].groupby("Year")["wind_speed_10m"].mean()
    ax.plot(sub.index, sub.values, marker="o", linewidth=2,
            label=station, color=colors[i], linestyle=linestyles[i])

ax.set_title("Mean Wind Speed Over Years by Station")
ax.set_xlabel("Year")
ax.set_ylabel("Mean Wind Speed (m/s)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 7. ZERO & NEGATIVE VALUES
# ============================================================
print("\n" + "=" * 60)
print("7. ZERO & NEGATIVE VALUES")
print("=" * 60)
for col in ["wind_speed_10m", "wind_direction_10m"]:
    zeros = (wind_df[col] == 0).sum()
    negs = (wind_df[col] < 0).sum()
    nulls = wind_df[col].isna().sum()
    print(f"  {col}: zeros={zeros:,} ({zeros/len(wind_df)*100:.2f}%) | "
          f"negatives={negs:,} | nulls={nulls:,}")

In [ ]:
# 8. DUPLICATES
# ============================================================
print("\n" + "=" * 60)
print("8. DUPLICATES")
print("=" * 60)
print(f"Exact duplicate rows: {wind_df.duplicated().sum()}")
print(f"Duplicates by station + date: {wind_df.duplicated(subset=['station', 'date']).sum()}")

In [ ]:
print("\n" + "=" * 60)
print("9. DATA QUALITY SUMMARY")
print("=" * 60)
print(f"  Total rows: {len(wind_df):,}")
print(f"  Columns: {wind_df.shape[1]}")
print(f"  Columns with nulls: {(wind_df.isna().sum() > 0).sum()}")
print(f"  Exact duplicates: {wind_df.duplicated().sum()}")
print(f"  Unique stations: {wind_df['station'].nunique()}")
print(f"  Date range: {wind_df['date'].min()} to {wind_df['date'].max()}")
print(f"  Wind speed range: {wind_df['wind_speed_10m'].min():.2f} to {wind_df['wind_speed_10m'].max():.2f} m/s")
print(f"  Wind direction range: {wind_df['wind_direction_10m'].min():.1f} to {wind_df['wind_direction_10m'].max():.1f} degrees")

In [ ]:
wind_df.head()

In [ ]:
# Filter wind data to 2021-2024 to match air quality data range
wind_df["date"] = pd.to_datetime(wind_df["date"]).dt.tz_localize(None)
wind_df = wind_df[wind_df["date"].dt.year.between(2021, 2024)].copy()

print(f"Wind rows after filtering: {len(wind_df):,}")
print(f"Date range: {wind_df['date'].min()} to {wind_df['date'].max()}")

####BURASI


In [ ]:
df_merged = pd.read_csv("df_merged_stats_parquet_pre.csv")

In [ ]:
df_merged.head()

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get unique stations from both datasets
aq_stations = df_merged[["Air Quality Station Name", "Latitude", "Longitude"]].drop_duplicates()
wind_stations = wind_df[["station", "latitude", "longitude"]].drop_duplicates()

print("Matching air quality stations to closest wind stations:\n")

for _, aq in aq_stations.iterrows():
    min_dist = float("inf")
    closest = None
    for _, ws in wind_stations.iterrows():
        d = haversine(aq["Latitude"], aq["Longitude"], ws["latitude"], ws["longitude"])
        if d < min_dist:
            min_dist = d
            closest = ws["station"]
    print(f"  {aq['Air Quality Station Name']:20s} -> {closest:20s} ({min_dist:.2f} km)")

In [ ]:
wind_df.head()

In [ ]:
df_merged.head()

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get unique wind stations
wind_stations = wind_df[["station", "latitude", "longitude"]].drop_duplicates()

# For each air quality station, find closest wind station
aq_stations = df_merged[["Air Quality Station Name", "Latitude", "Longitude"]].drop_duplicates()

station_mapping = {}
for _, aq in aq_stations.iterrows():
    min_dist = float("inf")
    closest = None
    for _, ws in wind_stations.iterrows():
        d = haversine(aq["Latitude"], aq["Longitude"], ws["latitude"], ws["longitude"])
        if d < min_dist:
            min_dist = d
            closest = ws["station"]
    station_mapping[aq["Air Quality Station Name"]] = closest

# Add closest wind station column to df_merged
df_merged["wind_station"] = df_merged["Air Quality Station Name"].map(station_mapping)

print("Station mapping:")
for aq, ws in station_mapping.items():
    print(f"  {aq} -> {ws}")

In [ ]:
df_merged.head()

In [ ]:
wind_df.head()

In [ ]:
df_merged["Start"] = pd.to_datetime(df_merged["Start"], format="mixed")
wind_df["date"] = pd.to_datetime(wind_df["date"], format="mixed")

In [ ]:
# Make sure both datetimes are aligned
df_merged["Start"] = pd.to_datetime(df_merged["Start"])
wind_df["date"] = pd.to_datetime(wind_df["date"])

# Round both to nearest hour to ensure matching
df_merged["hour_key"] = df_merged["Start"].dt.floor("H")
wind_df["hour_key"] = wind_df["date"].dt.floor("H")

# Merge: match wind station + hour
df_wind_merged = pd.merge(
    df_merged,
    wind_df[["station", "hour_key", "wind_speed_10m", "wind_direction_10m"]],
    left_on=["wind_station", "hour_key"],
    right_on=["station", "hour_key"],
    how="left"
)

# Drop extra columns
df_wind_merged = df_wind_merged.drop(columns=["station", "hour_key"])

matched = df_wind_merged["wind_speed_10m"].notna().sum()
total = len(df_wind_merged)
print(f"Rows: {total:,}")
print(f"Wind data matched: {matched:,} ({matched/total*100:.1f}%)")

In [ ]:
df_wind_merged.head()

In [ ]:
df_wind_merged = df_wind_merged.to_csv("air_wind_last_.csv")

In [ ]:
dataset1 = pd.read_csv("air_wind_last_.csv")

In [ ]:
dataset1.columns

In [ ]:
non_numeric = dataset1.select_dtypes(include="object").columns
for col in non_numeric:
    print(f"  {col}: {dataset1[col].nunique()} unique")

#last eda


In [ ]:
dataset5 = pd.read_csv("df_merged_stats_parquet_pre.csv")

In [ ]:
lastdataframe = pd.read_csv("air_wind_last_.csv")

In [ ]:
lastdataframe.head()

In [ ]:
lastdataframe.columns

In [ ]:
print(f"Shape: {lastdataframe.shape}")
print(f"\nUnique stations: {lastdataframe['Air Quality Station Name'].nunique()}")
print(lastdataframe["Air Quality Station Name"].value_counts())

In [ ]:
lastwind = pd.read_csv("wind_data_lazio.csv")

In [ ]:
lastwind.head()

In [ ]:
dataframe10 = pd.read_csv("df_merged_stats_parquet_pre.csv")

In [ ]:
dataframe10.head()

In [ ]:
print(f"Shape: {dataframe10.shape}")
print(f"\nUnique stations: {dataframe10['Air Quality Station Name'].nunique()}")
print(dataframe10["Air Quality Station Name"].value_counts())

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# Get unique stations from both datasets
aq_stations = dataframe10[["Air Quality Station Name", "Latitude", "Longitude"]].drop_duplicates()
wind_stations = lastwind[["station", "latitude", "longitude"]].drop_duplicates()

# Map each AQ station to closest wind station
station_mapping = {}
for _, aq in aq_stations.iterrows():
    min_dist = float("inf")
    closest = None
    for _, ws in wind_stations.iterrows():
        d = haversine(aq["Latitude"], aq["Longitude"], ws["latitude"], ws["longitude"])
        if d < min_dist:
            min_dist = d
            closest = ws["station"]
    station_mapping[aq["Air Quality Station Name"]] = closest

# Add to dataframe10
dataframe10["wind_station"] = dataframe10["Air Quality Station Name"].map(station_mapping)

print("Station mapping:")
for aq, ws in station_mapping.items():
    print(f"  {aq} -> {ws}")

In [ ]:
# Align datetimes
dataframe10["Start"] = pd.to_datetime(dataframe10["Start"], format="mixed")
lastwind["date"] = pd.to_datetime(lastwind["date"], format="mixed").dt.tz_localize(None)

# Create hour key for matching
dataframe10["hour_key"] = dataframe10["Start"].dt.floor("H")
lastwind["hour_key"] = lastwind["date"].dt.floor("H")

# Merge wind data
dataframe10 = pd.merge(
    dataframe10,
    lastwind[["station", "hour_key", "wind_speed_10m", "wind_direction_10m"]],
    left_on=["wind_station", "hour_key"],
    right_on=["station", "hour_key"],
    how="left"
)

dataframe10 = dataframe10.drop(columns=["station", "hour_key"])

matched = dataframe10["wind_speed_10m"].notna().sum()
print(f"\nWind data matched: {matched:,} / {len(dataframe10):,} ({matched/len(dataframe10)*100:.1f}%)")

In [ ]:
matched

In [ ]:
dataframe10

In [ ]:
dataframe10.columns

In [ ]:
# Unnamed: 0 — index artifact from CSV save
# End — redundant, Start + 1 hour
# Unit — same unit for all rows
# Verification — already cleaned using Validity
# Air Pollutant Description — redundant with Pollutant code
# Samplingpoint — station name is more readable
dataframe10 = dataframe10.drop(columns=[
    "Unnamed: 0", "End", "Unit", "Verification",
    "Air Pollutant Description", "Samplingpoint"
])

print(f"Remaining columns: {dataframe10.columns.tolist()}")

In [ ]:
dataframe10 = dataframe10.drop(columns = "Validity")

In [ ]:
dataframe10 = dataframe10.drop(columns = "Start")

In [ ]:
dataframe10 = dataframe10.to_csv("air&wind_combined.csv")

In [ ]:
dataframe10 = pd.read_csv("air&wind_combined.csv")

In [ ]:
"""
EDA - Air Quality & Wind Relationship
"""
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ============================================================
# 1. BASIC CHECK
# ============================================================
print(f"Shape: {dataframe10.shape}")
print(f"Unique stations: {dataframe10['Air Quality Station Name'].nunique()}")
print(f"Unique pollutants: {dataframe10['Pollutant'].nunique()}")
print(f"Year range: {dataframe10['Year'].min()} - {dataframe10['Year'].max()}")
print(f"\nNulls:")
print(dataframe10.isna().sum())

# ============================================================
# 2. WIND SPEED vs POLLUTION - Scatter
# ============================================================
# Filter valid data
valid = dataframe10[dataframe10["Value"].notna() & dataframe10["wind_speed_10m"].notna()]

top_pollutants = valid["Pollutant"].value_counts().head(4).index

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, poll in enumerate(top_pollutants):
    sub = valid[valid["Pollutant"] == poll]
    axes[i].scatter(sub["wind_speed_10m"], sub["Value"], alpha=0.05, s=1)
    axes[i].set_title(f"{poll}")
    axes[i].set_xlabel("Wind Speed (m/s)")
    axes[i].set_ylabel("Pollution Value")
    axes[i].set_ylim(0, sub["Value"].quantile(0.99))

plt.suptitle("Wind Speed vs Pollution Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ============================================================
# 3. WIND SPEED BINS vs POLLUTION - Boxplot
# ============================================================
valid["wind_bin"] = pd.cut(valid["wind_speed_10m"], bins=[0, 3, 6, 10, 15, 50],
                            labels=["0-3", "3-6", "6-10", "10-15", "15+"])

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, poll in enumerate(top_pollutants):
    sub = valid[valid["Pollutant"] == poll]
    sub.boxplot(column="Value", by="wind_bin", ax=axes[i], showfliers=False,
                patch_artist=True, boxprops=dict(facecolor="steelblue", alpha=0.7),
                medianprops=dict(color="red", linewidth=2))
    axes[i].set_title(f"{poll}")
    axes[i].set_xlabel("Wind Speed Bin (m/s)")
    axes[i].set_ylabel("Pollution Value")
    plt.suptitle("")

plt.suptitle("Pollution Level by Wind Speed Range", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# ============================================================
# 4. WIND DIRECTION vs POLLUTION - Polar plot
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 12), subplot_kw={"projection": "polar"})
axes = axes.flatten()

direction_bins = np.linspace(0, 360, 13)  # 12 sectors of 30 degrees
labels = ["N", "NNE", "ENE", "E", "ESE", "SSE", "S", "SSW", "WSW", "W", "WNW", "NNW"]

for i, poll in enumerate(top_pollutants):
    sub = valid[valid["Pollutant"] == poll]
    sub_dir = sub.copy()
    sub_dir["dir_bin"] = pd.cut(sub_dir["wind_direction_10m"], bins=direction_bins, labels=labels)
    mean_by_dir = sub_dir.groupby("dir_bin")["Value"].mean()

    theta = np.deg2rad(np.linspace(0, 330, 12) + 15)
    axes[i].bar(theta, mean_by_dir.values, width=np.deg2rad(30), color="red", alpha=0.7)
    axes[i].set_theta_zero_location("N")
    axes[i].set_theta_direction(-1)
    axes[i].set_title(f"{poll}", pad=20)

plt.suptitle("Mean Pollution by Wind Direction", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# ============================================================
# 5. CORRELATION MATRIX
# ============================================================
corr_data = valid[["Value", "wind_speed_10m", "wind_direction_10m", "Hour", "Altitude"]].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr_data, annot=True, cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation Matrix")
plt.tight_layout()
plt.show()

# ============================================================
# 6. HOURLY PATTERN - Pollution vs Wind
# ============================================================
hourly_avg = valid.groupby("Hour").agg(
    pollution=("Value", "mean"),
    wind_speed=("wind_speed_10m", "mean")
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 5))
ax2 = ax1.twinx()

ax1.plot(hourly_avg["Hour"], hourly_avg["pollution"], "b-o", label="Mean Pollution")
ax2.plot(hourly_avg["Hour"], hourly_avg["wind_speed"], "r-o", label="Mean Wind Speed")

ax1.set_xlabel("Hour of Day")
ax1.set_ylabel("Mean Pollution Value", color="blue")
ax2.set_ylabel("Mean Wind Speed (m/s)", color="red")
ax1.set_title("Hourly Pattern: Pollution vs Wind Speed")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Group wind direction into 8 compass directions
def get_compass(deg):
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    idx = int(((deg + 22.5) % 360) / 45)
    return directions[idx]

valid = dataframe10[dataframe10["Value"].notna() & dataframe10["wind_direction_10m"].notna()].copy()
valid["compass"] = valid["wind_direction_10m"].apply(get_compass)

# Order for plotting
compass_order = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

top_pollutants = valid["Pollutant"].value_counts().head(6).index

# 1. Mean pollution by compass direction per pollutant
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, poll in enumerate(top_pollutants):
    sub = valid[valid["Pollutant"] == poll]
    means = sub.groupby("compass")["Value"].mean().reindex(compass_order)
    axes[i].bar(compass_order, means.values, color="steelblue", edgecolor="black")
    axes[i].set_title(f"{poll}", fontsize=12, fontweight="bold")
    axes[i].set_ylabel("Mean Value")
    axes[i].set_xlabel("Wind Direction")

plt.suptitle("Mean Pollution Level by Wind Direction", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

# 2. Polar version - same data but on compass rose
fig, axes = plt.subplots(2, 3, figsize=(18, 12), subplot_kw={"projection": "polar"})
axes = axes.flatten()

angles = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])

for i, poll in enumerate(top_pollutants):
    sub = valid[valid["Pollutant"] == poll]
    means = sub.groupby("compass")["Value"].mean().reindex(compass_order)

    # Close the circle
    vals = list(means.values) + [means.values[0]]
    theta = list(angles) + [angles[0]]

    axes[i].fill(theta, vals, alpha=0.3, color="red")
    axes[i].plot(theta, vals, "o-", color="red", linewidth=2)
    axes[i].set_theta_zero_location("N")
    axes[i].set_theta_direction(-1)
    axes[i].set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315], compass_order)
    axes[i].set_title(f"{poll}", pad=20, fontsize=12, fontweight="bold")

plt.suptitle("Pollution Level by Wind Direction (Compass Rose)", fontsize=16, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# 3. Summary table
print("\nMean pollution by wind direction:")
summary = valid.groupby(["compass", "Pollutant"])["Value"].mean().unstack()
summary = summary.reindex(compass_order)
print(summary.round(2).to_string())

In [ ]:
df_factories

In [ ]:
df_factories = pd.read_csv("F1_4_Air_Releases_Facilities.csv")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import contextily as ctx
import pandas as pd

# Rome center
rome_lat, rome_lon = 41.9028, 12.4964

# Get factory locations and drop rows with missing coordinates
factories = (
    df_factories[["Latitude", "Longitude", "EPRTR_SectorName"]]
    .drop_duplicates()
    .dropna(subset=["Latitude", "Longitude"])
    .copy()
)

# Get AQ station locations and drop rows with missing coordinates
aq_stations = (
    dataframe10[["Air Quality Station Name", "Latitude", "Longitude"]]
    .drop_duplicates()
    .dropna(subset=["Latitude", "Longitude"])
    .copy()
)

# Calculate direction from Rome to each factory
def get_bearing(lat1, lon1, lat2, lon2):
    if pd.isna(lat2) or pd.isna(lon2):
        return np.nan
    dlon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(x, y))
    return (bearing + 360) % 360

factories["bearing_from_rome"] = factories.apply(
    lambda r: get_bearing(rome_lat, rome_lon, r["Latitude"], r["Longitude"]), axis=1
)

def get_compass(deg):
    if pd.isna(deg):
        return np.nan
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    idx = int(((deg + 22.5) % 360) / 45)
    return directions[idx]

factories["direction_from_rome"] = factories["bearing_from_rome"].apply(get_compass)

print("Factories by direction from Rome:")
print(
    factories["direction_from_rome"]
    .value_counts()
    .reindex(["N", "NE", "E", "SE", "S", "SW", "W", "NW"], fill_value=0)
)

print("\nFactories by sector and direction:")
print(
    factories.groupby(["direction_from_rome", "EPRTR_SectorName"])
    .size()
    .unstack(fill_value=0)
)



In [ ]:
df_factories.columns

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import radians, sin, cos, atan2, degrees
# Drop rows with missing coordinates before calculating direction
df_factories = df_factories.dropna(subset=["Latitude", "Longitude"])

# Updated get_compass with NaN safety
def get_compass(deg):
    if pd.isna(deg):
        return "Unknown"
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    idx = int(((deg + 22.5) % 360) / 45)
    return directions[idx]
# Rome center
rome_lat, rome_lon = 41.9028, 12.4964

# Calculate bearing (direction) from Rome to each factory
def get_bearing(lat1, lon1, lat2, lon2):
    dlon = np.radians(lon2 - lon1)
    lat1, lat2 = np.radians(lat1), np.radians(lat2)
    x = np.sin(dlon) * np.cos(lat2)
    y = np.cos(lat1) * np.sin(lat2) - np.sin(lat1) * np.cos(lat2) * np.cos(dlon)
    bearing = np.degrees(np.arctan2(x, y))
    return (bearing + 360) % 360

def get_compass(deg):
    directions = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]
    idx = int(((deg + 22.5) % 360) / 45)
    return directions[idx]

df_factories["bearing_from_rome"] = df_factories.apply(
    lambda r: get_bearing(rome_lat, rome_lon, r["Latitude"], r["Longitude"]), axis=1
)
df_factories["direction_from_rome"] = df_factories["bearing_from_rome"].apply(get_compass)

compass_order = ["N", "NE", "E", "SE", "S", "SW", "W", "NW"]

# 1. Number of factories by direction
print("Factory count by direction from Rome:")
print(df_factories["direction_from_rome"].value_counts().reindex(compass_order))

# 2. Total releases by direction
releases_by_dir = df_factories.groupby("direction_from_rome")["Releases"].sum().reindex(compass_order)
print("\nTotal releases (kg) by direction:")
print(releases_by_dir)

# 3. Bar chart: factory count by direction
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_factories["direction_from_rome"].value_counts().reindex(compass_order).plot(
    kind="bar", ax=axes[0], color="steelblue", edgecolor="black")
axes[0].set_title("Number of Factories by Direction from Rome", fontweight="bold")
axes[0].set_ylabel("Count")
axes[0].set_xlabel("Direction")

releases_by_dir.plot(kind="bar", ax=axes[1], color="red", edgecolor="black")
axes[1].set_title("Total Releases by Direction from Rome", fontweight="bold")
axes[1].set_ylabel("Total Releases (kg)")
axes[1].set_xlabel("Direction")

plt.tight_layout()
plt.show()

# 4. Polar plot: releases by direction
fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={"projection": "polar"})
angles = np.deg2rad([0, 45, 90, 135, 180, 225, 270, 315])

# Factory count polar
counts = df_factories["direction_from_rome"].value_counts().reindex(compass_order).values
vals = list(counts) + [counts[0]]
theta = list(angles) + [angles[0]]
axes[0].fill(theta, vals, alpha=0.3, color="steelblue")
axes[0].plot(theta, vals, "o-", color="steelblue", linewidth=2)
axes[0].set_theta_zero_location("N")
axes[0].set_theta_direction(-1)
axes[0].set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315], compass_order)
axes[0].set_title("Factory Count by Direction", pad=20, fontweight="bold")

# Releases polar
rels = releases_by_dir.values
vals = list(rels) + [rels[0]]
axes[1].fill(theta, vals, alpha=0.3, color="red")
axes[1].plot(theta, vals, "o-", color="red", linewidth=2)
axes[1].set_theta_zero_location("N")
axes[1].set_theta_direction(-1)
axes[1].set_thetagrids([0, 45, 90, 135, 180, 225, 270, 315], compass_order)
axes[1].set_title("Total Releases by Direction", pad=20, fontweight="bold")

plt.tight_layout()
plt.show()

# 5. Releases by direction AND sector
print("\nReleases by direction and sector:")
sector_dir = df_factories.groupby(["direction_from_rome", "EPRTR_SectorName"])["Releases"].sum().unstack(fill_value=0)
sector_dir = sector_dir.reindex(compass_order)
print(sector_dir.round(0).to_string())

# 6. Compare with pollution compass rose
# If you already have the pollution compass data from earlier:
print("\n\nCOMPARISON:")
print("If factory releases are highest from direction X,")
print("and pollution is highest when wind comes FROM direction X,")
print("that suggests factories in that direction are contributing to Rome's air quality.")

In [ ]:
# Focus on E and SE — where most pollution comes from
high_pollution_dirs = df_factories[df_factories["direction_from_rome"].isin(["E", "SE"])].copy()

print(f"Factories in E + SE: {len(high_pollution_dirs)}")
print(f"Total releases E+SE: {high_pollution_dirs['Releases'].sum():,.0f} kg")
print(f"\n--- Sector breakdown ---")
print(high_pollution_dirs.groupby("EPRTR_SectorName")["Releases"].agg(["count", "sum", "mean"]).sort_values("sum", ascending=False).round(0))

print(f"\n--- Top pollutants ---")
print(high_pollution_dirs.groupby("Pollutant")["Releases"].sum().sort_values(ascending=False).head(10))

print(f"\n--- Top facilities by releases ---")
top_fac = high_pollution_dirs.groupby(["EPRTRAnnexIMainActivity", "Latitude", "Longitude"])["Releases"].sum().sort_values(ascending=False).head(10)
print(top_fac)

# Compare: releases per factory by direction
print(f"\n--- Mean releases per factory by direction ---")
per_factory = df_factories.groupby("direction_from_rome").agg(
    num_factories=("Releases", "count"),
    total_releases=("Releases", "sum"),
    mean_releases=("Releases", "mean")
).reindex(["N", "NE", "E", "SE", "S", "SW", "W", "NW"])
print(per_factory.round(0))

# Plot: compare NW (most factories) vs E/SE (most pollution)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Releases per factory by direction
per_factory["mean_releases"].plot(kind="bar", ax=axes[0], color="red", edgecolor="black")
axes[0].set_title("Mean Releases PER FACTORY by Direction", fontweight="bold")
axes[0].set_ylabel("Mean Releases (kg)")
axes[0].set_xlabel("Direction")

# Sector comparison: NW vs E/SE
nw = df_factories[df_factories["direction_from_rome"] == "NW"]
ese = df_factories[df_factories["direction_from_rome"].isin(["E", "SE"])]

compare = pd.DataFrame({
    "NW": nw.groupby("EPRTR_SectorName")["Releases"].sum(),
    "E+SE": ese.groupby("EPRTR_SectorName")["Releases"].sum()
}).fillna(0)

compare.plot(kind="barh", ax=axes[1], color={"NW": "steelblue", "E+SE": "red"})
axes[1].set_title("Releases by Sector: NW vs E+SE", fontweight="bold")
axes[1].set_xlabel("Total Releases (kg)")

plt.tight_layout()
plt.show()

In [ ]:
# Energy sector releases over years
energy = df_factories[df_factories["EPRTR_SectorName"] == "Energy sector"]

# Total and mean releases by year
yearly_energy = energy.groupby("reportingYear")["Releases"].agg(["sum", "mean", "count"]).reset_index()
yearly_energy.columns = ["Year", "Total_Releases", "Mean_Releases", "Facility_Count"]

print(yearly_energy.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(yearly_energy["Year"], yearly_energy["Total_Releases"], marker="o", color="red", linewidth=2)
axes[0].set_title("Total Energy Sector Releases", fontweight="bold")
axes[0].set_ylabel("Total Releases (kg)")
axes[0].set_xlabel("Year")

axes[1].plot(yearly_energy["Year"], yearly_energy["Mean_Releases"], marker="o", color="darkorange", linewidth=2)
axes[1].set_title("Mean Releases per Facility", fontweight="bold")
axes[1].set_ylabel("Mean Releases (kg)")
axes[1].set_xlabel("Year")

axes[2].bar(yearly_energy["Year"], yearly_energy["Facility_Count"], color="steelblue", edgecolor="black")
axes[2].set_title("Number of Reporting Facilities", fontweight="bold")
axes[2].set_ylabel("Count")
axes[2].set_xlabel("Year")

plt.suptitle("Energy Sector Trends", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# By pollutant for energy sector
fig, ax = plt.subplots(figsize=(12, 6))
top_polls = energy.groupby("Pollutant")["Releases"].sum().sort_values(ascending=False).head(5).index

for poll in top_polls:
    sub = energy[energy["Pollutant"] == poll].groupby("reportingYear")["Releases"].sum()
    ax.plot(sub.index, sub.values, marker="o", linewidth=2, label=poll)

ax.set_yscale("log")
ax.set_title("Energy Sector Releases by Pollutant Over Years (Log Scale)", fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Total Releases (kg, log)")
ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()